# Whisper Audio Transcriber

Upload one or multiple audio files and download matching `.txt` transcripts.

If multiple files are uploaded, all transcripts are packaged into `transcripts.zip`.


In [ ]:
!pip -q install openai-whisper
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
print('Whisper and FFmpeg are ready.')

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU. For faster transcription, choose Runtime > Change runtime type > T4 GPU.')

In [ ]:
from google.colab import files
import whisper
import os
import time
import zipfile
import shutil

uploaded = files.upload()
audio_files = list(uploaded.keys())

if not audio_files:
    raise ValueError('No audio files were uploaded.')

print(f'\n{len(audio_files)} file(s) uploaded.')

MODEL_NAME = 'base'
print(f'Loading Whisper model: {MODEL_NAME}')
model = whisper.load_model(MODEL_NAME)

output_dir = 'transcripts'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

completed_files = []
failed_files = []
total_start = time.time()

for number, audio_file in enumerate(audio_files, start=1):
    print('\n' + '=' * 70)
    print(f'File {number}/{len(audio_files)}')
    print(f'Processing: {audio_file}')
    print('=' * 70)
    start_time = time.time()
    try:
        result = model.transcribe(audio_file, verbose=True)
        base_name = os.path.splitext(os.path.basename(audio_file))[0]
        transcript_file = base_name + '.txt'
        transcript_path = os.path.join(output_dir, transcript_file)
        with open(transcript_path, 'w', encoding='utf-8') as f:
            f.write(result['text'].strip())
        elapsed = time.time() - start_time
        completed_files.append(transcript_file)
        print(f'\nCompleted: {audio_file}')
        print(f'Saved: {transcript_file}')
        print(f'Processing time: {elapsed/60:.1f} minutes')
    except Exception as e:
        failed_files.append((audio_file, str(e)))
        print(f'\nFailed: {audio_file}')
        print(f'Error: {e}')

total_elapsed = time.time() - total_start
print('\n' + '=' * 70)
print('BATCH COMPLETE')
print('=' * 70)
print(f'Successful: {len(completed_files)}')
print(f'Failed: {len(failed_files)}')
print(f'Total processing time: {total_elapsed/60:.1f} minutes')

if completed_files:
    zip_name = 'transcripts.zip'
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
        for transcript_file in completed_files:
            transcript_path = os.path.join(output_dir, transcript_file)
            z.write(transcript_path, arcname=transcript_file)
    print(f'\nCreated: {zip_name}')
    for name in completed_files:
        print(f'  - {name}')
    files.download(zip_name)

if failed_files:
    print('\nFiles that failed:')
    for name, error in failed_files:
        print(f'  - {name}: {error}')